## Integración con Langchain y langgraph

In [59]:
from dotenv import load_dotenv
import os

import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

from openai import OpenAI

import ast

from IPython.display import HTML, Markdown, display
import pandas as pd

# Add this line to load variables from .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')  # Retrieve the API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  # Retrieve the API key
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')  # Retrieve the API key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [60]:
! git branch

* db_f1
  master
  test_enc_1
  test_enc_2


In [61]:
# Install and configure ChromaDB with DuckDB+Parquet backend
from chromadb.config import Settings  
from chromadb import PersistentClient                             # CHANGED
from chromadb.config import Settings, DEFAULT_TENANT, DEFAULT_DATABASE  # CHANGED

# Reinitialize Chroma client with DuckDB+Parquet backend (new ChromaDB API)
from chromadb import Client
from chromadb.config import Settings

DB_PERSIST_DIR = "./db_f1"
DB_NAME        = "enc_dbf1"


client = PersistentClient(
    path     = DB_PERSIST_DIR,
    settings = Settings(allow_reset=True),   # allow resetting on-disk state
    tenant   = DEFAULT_TENANT,
    database = DEFAULT_DATABASE,
) 

print("Collections:", [c.name for c in client.list_collections()])  

db_f1 = client.get_or_create_collection(
    name               = DB_NAME,
    embedding_function = embedding_fun_openai
)

db_f1.peek()

# db_f1 = client.get_collection(name="enc_dbf1")

Collections: ['enc_dbf1']


{'ids': ['p1_1a_1|IDE__question',
  'p1_1a_1|IDE__summary',
  'p1_1a_1|IDE__implications',
  'p2_1a_1|IDE__question',
  'p2_1a_1|IDE__summary',
  'p2_1a_1|IDE__implications',
  'p3_1|IDE__question',
  'p3_1|IDE__summary',
  'p3_1|IDE__implications',
  'p3_2|IDE__question'],
 'embeddings': array([[-0.01286233, -0.02477617,  0.00845953, ..., -0.00216455,
         -0.01462473, -0.00672276],
        [-0.00771893,  0.00366222,  0.03773989, ..., -0.00310992,
         -0.00857366, -0.0436573 ],
        [-0.02147629,  0.00293677,  0.03042583, ...,  0.00184429,
         -0.02441142, -0.04355532],
        ...,
        [ 0.01838034,  0.00962718,  0.02427666, ..., -0.00525712,
         -0.02070235, -0.05458009],
        [-0.00535758,  0.0059622 ,  0.01842068, ..., -0.00686241,
         -0.02355321, -0.02821548],
        [-0.00732108, -0.0118375 ,  0.03220749, ..., -0.00837447,
         -0.01164658, -0.00516163]], shape=(10, 1536)),
 'documents': ['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio

In [62]:
# clientes para encoding de query, etc

client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [63]:
import pickle

ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'

with open(f'{ruta_enc}/los_mex_dict.pkl', 'rb') as f:
    los_mex_dict = pickle.load(f)
    print('los_mex_dict cargado --  leer readme_dict para info')


enc_dict = los_mex_dict['enc_dict']
enc_nom_dict = los_mex_dict['enc_nom_dict']
rev_enc_nom_dict = {v: k for k, v in enc_nom_dict.items()}

pregs_dict = los_mex_dict['pregs_dict']
ses_dict = los_mex_dict['ses_dict']
mkdown_tables = los_mex_dict['mkdown_tables']
df_tables = los_mex_dict['df_tables']

# TODO: usar df_tables para plots

los_mex_dict cargado --  leer readme_dict para info


### query v2: creación de resúmenes temáticos

In [64]:
# query de la base de embeddings
# query_emb es el embedding de la pregunta

user_query= '¿qué significa ser mexicano para los mexicanos?'
# turn it into a vector
query_emb = embedding_fun_openai([user_query])[0]
query_emb

array([-0.01175261, -0.02690859,  0.01918369, ...,  0.00272391,
       -0.02097107, -0.0183757 ], shape=(1536,), dtype=float32)

In [65]:
# ejemplo: query entre las preguntas

result_q = db_f1.query(
    query_embeddings = [query_emb],
    n_results        = 5,
    where            = {"type": "question"}
)

print(result_q["documents"])   # the matched question strings
print(result_q["metadatas"])   # metadata for each hit (includes qid & type)


[['CULTURA_POLITICA|¿Qué tan orgulloso se siente de ser mexicano?', 'IDENTIDAD_Y_VALORES|¿Qué tan orgulloso se siente de ser mexicano?', 'GLOBALIZACION|¿Qué tan orgulloso está de ser mexicano?', 'FEDERALISMO|¿Qué tan orgulloso se siente usted de ser mexicano?', 'INDIGENAS|¿Cuál cree que es la mayor ventaja de ser indígena en México?']]
[[{'type': 'question', 'qid': 'p5|CUL'}, {'type': 'question', 'qid': 'p6|IDE'}, {'qid': 'p15|GLO', 'type': 'question'}, {'type': 'question', 'qid': 'p19_3|FED'}, {'type': 'question', 'qid': 'p13a_1|IND'}]]


In [66]:
# tmp_dist_df contiene las distancias para los tres tipos/ facets, normalizadas entre 0 y 1

# OJO: esto obviamente asume que los tres tipos de información tienen la misma importancia, lo cual no es inicialmente cierto.
# Pero la mezcla de las tres calificaciones devuelve una variedad más amplia de resultados.  

type_lst = [ "question", "summary", "implications"]

tmp_dist_dict = {}
for type in type_lst:
    print(f"Querying for type: {type}")
    tmp_res_q = db_f1.query(
        query_embeddings = [query_emb],
        n_results        = 100,  # devuelvo 100 resultados para cada tipo con distancias menores
        where            = {"type": type}
    )
    [tmp_res_ids] = tmp_res_q['ids']
    [tmp_res_distances] = tmp_res_q['distances']

    tmp_dist_dict[type]= dict(zip(tmp_res_ids, tmp_res_distances))

# remove the suffixes from the keys
tmp_dist_dict = { outer_key: { k.split('__')[0]: v for k, v in inner_dict.items() }
    for outer_key, inner_dict in tmp_dist_dict.items() }

# Create a DataFrame where keys in every subdict are the index and keys in tmp_dist_dict are columns
tmp_dist_df = pd.DataFrame.from_dict(tmp_dist_dict)

# Normalize each column so that max = 1 and min = 0
tmp_dist_df = (tmp_dist_df - tmp_dist_df.min()) / (tmp_dist_df.max() - tmp_dist_df.min())

Querying for type: question
Querying for type: summary
Querying for type: implications


In [67]:
# tmp_res_st contiene las 40 preguntas más cercanas y la descripción de sus resultados.

top_vals = 40

tmp_dist_df['mean'] = tmp_dist_df.mean(axis=1)
tmp_dist_df.sort_values(by='mean', ascending=True, inplace=True)

top_ids = tmp_dist_df.head(top_vals).index.tolist()

tmp_list = []

for type in type_lst:
    for id in top_ids:
        tmp_list.append(id + f'__{type}')


# Retrieve documents using the list of ids
result_by_ids = db_f1.get(ids=tmp_list)

tmp_pre_res_dict = dict(zip(result_by_ids['ids'], result_by_ids['documents']))

tmp_preproc_dic = {k: v for k, v in tmp_pre_res_dict.items() if k.split('__')[1] in ['question', 'summary'] }

# # instead of your current list-comp do this:
tmp_combined_strings = []

 # Calculate the group index based on the position

for i, (k, v) in enumerate(tmp_preproc_dic.items(), start=1):
    facet = k.split("__", 1)[1].upper()
    p_id = k#.split("__")[0].split("|")[0]

    grouped_index = (i + 1) // 2 
    # only split once, and if there is no "|" take the entire v
    q_id = v.split("|")[0]
    parts = v.split("|", 1)
    text = parts[1] if len(parts) > 1 else parts[0]

    p_id = p_id.split("__")[0]

    tmp_combined_strings.append(f"{facet}_{grouped_index}|{p_id}: {text}")

tmp_res_st = '\n'.join(tmp_combined_strings)
tmp_res_st

"QUESTION_1|p1_1a_1|IDE: Con la palabra maíz, yo asocio comida, mercado, animales. Dígame por favor, tres palabras que asocies con la palabra MÉXICO. 1° MENCIÓN\nSUMMARY_1|p1_1a_1|IDE: The table presents various associations that people have with the word 'México', showcasing a wide range of responses. The most prominent association is 'Orgullo', which reflects a strong sense of national pride at 25.25%, followed by 'Costumbres y tradiciones' at 13.50%, and 'Historia y cultura' at 7.58%. Notably, the 'No sabe/ No contesta' response is at 9.75%, indicating a significant portion of respondents either do not have an association or are unwilling to share. This highlights potential gaps in connection or sentiment towards the national identity.\nQUESTION_2|p2_1a_1|IDE: Y ahora, voy a pedir que me diga, por favor, tres palabras que asocie con la palabra MEXICANO. 1° MENCIÓN\nSUMMARY_2|p2_1a_1|IDE: The table presents various words and phrases associated with the identity of 'Mexicano', indicat

In [68]:
## generación del esquema de pydantic para la respuesta

from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# 1. Define Pydantic model for an entry
class PatternSummaryEntry(BaseModel):
    SIMILAR_PATTERN_1: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_2: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_3: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_4: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    SIMILAR_PATTERN_5: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_1: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_2: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_3: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_4: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }
    DIFFERENT_PATTERN_5: dict[str, str] = {
        "TITLE_SUMMARY": "",
        "VARIABLE_STRING": "", 
        "DESCRIPTION": ""
    }

# 2. Create parser for single entries
pattern_parser_summary = PydanticOutputParser(pydantic_object=PatternSummaryEntry)
pattern_format_instructions = pattern_parser_summary.get_format_instructions()

In [77]:
# propmt optiizado para usar el parser de pydantic

def create_prompt_crosssum(user_query, tmp_res_st, n_topics=5, format_instructions=""):
    """
    Creates a prompt for analyzing a table and answering a query.

    Parameters:
        user_query (str): The query to be answered.
        tmp_res_st (str): The results to analyze.
        n_topics (int): The number of similar and different patterns to identify.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do its work in either language. But you will reply in English.

    Your task is to read the survey RESULTS and find the top {n_topics} SIMILAR PATTERNS across the RESULTS that best answer the QUERY below.
    Note SIMILAR PATTERNS are those that show agreement across different questions and summaries, and that are relevant to the QUERY. 
    You will also return {n_topics} DIFFERENT PATTERNS that are relevant to the QUERY, but that show a pattern that contradicts any of the SIMILAR PATTERNS.

    Note that the RESULTS will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT. 

    For example, in the following results:

    TEST_QUERY: 'Do people like sweets?'
    TEST_RESULTS:
    QUESTION_1|p1_1|TPC1: 'Do you like ice cream?'
    SUMMARY_1|p1_1|TPC1: 85% of people like ice cream, while 10% said they do not like it, and 5% said they do not know.
    QUESTION_2|p2|TPC1: 'Do you like marshmallow treats?'
    SUMMARY_2|p2|TPC1: 80% of people like marshmallow treats, while 15% said they do not like it, and 5% said they do not know.
    QUESTION_3|p10|TPC2: 'Do you like sour apple candies?'
    SUMMARY_3|p10|TPC2: 40% of people like sour apple candies, while 50% said they do not like it, and 10% said they do not know.

    In these cases, the SIMILAR PATTERN is that 'people really like sweet treats because 85% like ice cream and 80% like marshmallow treats', 
    and the DIFFERENT PATTERN is that 'people do not like sour treats as much because only 40% said they liked them'.

    You will describe the PATTERNS in a simple, but detailed way. 
    Be sure to include as many relevant facts (numbers) as possible and describe the pattern and the supporting facts. 
    Only include facts you are sure about, and for every fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).
    
    You will write them as a VARIABLE_STRING with format pxx|TYY (where x is a number and YY is a table id), but separating using ',' eg 'p1_1|TPC1,p2|TPC1,p10|TPC2'.
    Only include the QUESTION_IDs of the variables you used to create the PATTERN and that you mention in it.

    Note that any mention of results not available or 'NaN' is NOT THE SAME as 'Do not know' or 'no answer'. 
    You should ignore any reference to results that are not available or 'NaN'.

    If you discuss two similar categories together (e.g., 'like a lot' and 'like somehow'), only mention the sum of these percentages if you have all the original ones to sum them. Otherwise, do not mention the figure itself. 
    Do not discuss figures or percentages you are not sure about.

    Be careful to not be repetitive and to add a varied list of PATTERNS: if you have discussed any SIMILAR PATTERN or DIFFERENT PATTERN, do not add it to the list again, but come up with a different one. 
    
    QUERY: {user_query}
    RESULTS: {tmp_res_st.replace("\n", " ")}

    {format_instructions}
    """
    return prompt

# Create the prompt
prompt = create_prompt_crosssum(user_query="What does it mean to be Mexican?", tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_format_instructions)

print(prompt)


    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do its work in either language. But you will reply in English.

    Your task is to read the survey RESULTS and find the top 5 SIMILAR PATTERNS across the RESULTS that best answer the QUERY below.
    Note SIMILAR PATTERNS are those that show agreement across different questions and summaries, and that are relevant to the QUERY. 
    You will also return 5 DIFFERENT PATTERNS that are relevant to the QUERY, but that show a pattern that contradicts any of the SIMILAR PATTERNS.

    Note that the RESULTS will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTI

In [78]:
import tiktoken

def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    # This function sends the prompt to the LLM and retrieves the response.
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

def get_structured_summary(model_name: str = 'gpt-4o-mini-2024-07-18', temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.
    prompt =create_prompt_crosssum(user_query=user_query, tmp_res_st=tmp_res_st, n_topics=5, format_instructions=pattern_format_instructions)
    content = get_answer(prompt, model=model_name, temperature=temperature)
    parsed = pattern_parser_summary.parse(content)
    return parsed.model_dump()  # Returns the parsed response as a dictionary.

tst_lgc_dict = get_structured_summary(model_name = 'gpt-4o-mini-2024-07-18', temperature= 0.9)
tst_lgc_dict
# 


{'SIMILAR_PATTERN_1': {'TITLE_SUMMARY': 'Pride in Mexican Identity',
  'VARIABLE_STRING': 'p5_1|IDE,p6|IDE,p39|FED',
  'DESCRIPTION': "A significant portion of respondents express a strong sense of pride in their Mexican identity, with 63.42% feeling 'Mucho' (a lot) pride (p5_1|IDE), and 89.25% stating they are either 'very proud' or 'proud' to be Mexican (p39|FED). This indicates a widespread emotional connection to their national identity."},
 'SIMILAR_PATTERN_2': {'TITLE_SUMMARY': 'Cultural Associations with National Identity',
  'VARIABLE_STRING': 'p1_1a_1|IDE,p2_1a_1|IDE',
  'DESCRIPTION': "The top associations with 'México' include feelings of 'Orgullo' (pride) at 25.25% and 'Costumbres y tradiciones' (customs and traditions) at 13.50% (p1_1a_1|IDE), while the identity of being 'Mexicano' is predominantly linked to being 'Trabajador' (hardworking) at 13.67% (p2_1a_1|IDE). This reflects a shared cultural sentiment that emphasizes pride in tradition and work ethic."},
 'SIMILAR_PAT

In [92]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# Define the Pydantic model for the output
class ExpertSummaryResponse(BaseModel):
    EXPERT_REPLY: str

# Create the parser
expert_summary_parser = PydanticOutputParser(pydantic_object=ExpertSummaryResponse)

# Generate format instructions
expert_summary_format_instructions = expert_summary_parser.get_format_instructions()

In [109]:
def create_prompt_expt_smry(tst_lgc_dict, tmp_ky, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and survey summaries.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """

    ky = tmp_ky

    # Variables identified by the model
    tst_str_lst = tst_lgc_dict[ky]['VARIABLE_STRING'].split(',')
    tst_str_lst = [st + '__question' for st in tst_str_lst]
    print(f'Variables identified by the model: {tst_str_lst}')

    # Variables in the database
    tmp_db_var_lst = db_f1.get(ids=tst_str_lst)['ids']
    print(f'Variables in the database: {tmp_db_var_lst}')

    # Hallucinated variables
    tmp_hlc_var_lst = set(tst_str_lst) - set(tmp_db_var_lst)
    if tmp_hlc_var_lst:
        print(f'Hallucinated variables by the model: {tmp_hlc_var_lst}')

    # Implications for the identified variables
    tmp_imp_lst = [st.replace('question', 'implications') for st in tmp_db_var_lst]
    tmp_ky_lst = [st.split('__')[0] for st in tmp_imp_lst]
    tmp_imp_dict = dict(
        zip(
            tmp_ky_lst,
            db_f1.get(ids=tmp_imp_lst)['documents']
        )
    )

    imp_st = ' * '.join(tmp_imp_dict.values())
    tmp_smry = tst_lgc_dict[ky]['DESCRIPTION']

    # Generate the prompt
    prompt = f"""
    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.
    
    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
    Do not refer to the experts or to yourself in the first person, just write the paragraph as if it was a report.

    Note that the SURVEY SUMMARY will have QUESTION_IDS of the format 'question_id|table_id' (e.g., pxx|YYY where x, which may include strings like x_x where x is a number, and YYY is a table id). 
    They are not relevant to your answer, but you will need to keep track of these QUESTION_IDS of format pxx|YYY in your RETURN OBJECT. 

    Be sure to include as many relevant facts (numbers) as possible and for each fact you include, you will include the QUESTION_ID in parenthesis (e.g., pxx|TYY).

    EXPERT STATEMENTS: * {imp_st}
    SURVEY SUMMARY: {tmp_smry}

    {format_instructions}
    """
    return prompt

tmp_ky = 'SIMILAR_PATTERN_1'

# Generate the prompt
prompt = create_prompt_expt_smry(
    tst_lgc_dict=tst_lgc_dict,
    tmp_ky=tmp_ky,
    format_instructions=expert_summary_format_instructions
)

print(prompt)

Variables identified by the model: ['p5_1|IDE__question', 'p6|IDE__question', 'p39|FED__question']
Variables in the database: ['p5_1|IDE__question', 'p6|IDE__question', 'p39|FED__question']

    You are a very thorough research assistant that is working on a survey research project.
    The objective of this project is to study public opinion on a variety of topics.
    You are fully bilingual in English and Spanish, and can do your work in either language. But you will reply in English.

    Your task is to read the EXPERT STATEMENTS from one or more experts about what information they consider important to receive from the survey results. 
    Then you will read a SURVEY SUMMARY of some survey results and will write a one-paragraph reply to the experts, elaborating on how the results relate and illustrate their concerns. 
    Note that they are experts and expect a detailed and thorough reply. Reply to all of them in the same paragraph, but be sure to address all of their points. 
  

In [110]:
import tqdm
import tiktoken

def get_structured_expert_summary(tst_lgc_dict, tmp_ky, model_name="gpt-4o-mini-2024-07-18", temperature=0.9):
    """
    Generates a structured expert summary for a given key in the logical group dictionary.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        tmp_ky (str): Key for the current logical group.
        model_name (str): Name of the LLM model to use.
        temperature (float): Temperature setting for the LLM.

    Returns:
        dict: Parsed response as a dictionary.
    """
    # Generate the prompt
    prompt = create_prompt_expt_smry(
        tst_lgc_dict=tst_lgc_dict,
        tmp_ky=tmp_ky,
        format_instructions=expert_summary_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    
    # Parse the response
    parsed = expert_summary_parser.parse(response)
    
    # Ensure the parsed response is returned as a dictionary
    return parsed.model_dump()


def num_tokens_from_string(string: str, encoding_name: str) -> int:
    # Utility function to calculate the number of tokens in a string.
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Update the token count calculation in batch_documents to use num_tokens_from_string
# 8192 is the token limit for the model
def batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base"):
    # This function batches documents and their IDs while respecting the token limit.
    # It ensures that each batch does not exceed the maximum token limit,
    # which is crucial for efficient processing with the LLM.
    batches = []
    current_batch_docs = []
    current_batch_ids = []
    current_token_count = 0

    for doc, doc_id in zip(docs, ids):
        doc_token_count = num_tokens_from_string(doc, encoding_name)  # Use the token counting function
        
        # print(f"Document ID: {doc_id}, Token Count: {doc_token_count}")
        
        if current_token_count + doc_token_count > max_tokens:
            # Save the current batch and start a new one
            batches.append((current_batch_docs, current_batch_ids))
            current_batch_docs = []
            current_batch_ids = []
            current_token_count = 0

        # Add the document to the current batch
        current_batch_docs.append(doc)
        current_batch_ids.append(doc_id)
        current_token_count += doc_token_count

    # Add the last batch if it has any documents
    if current_batch_docs:
        batches.append((current_batch_docs, current_batch_ids))

    return batches

def batch_process_expert_summaries(tst_lgc_dict, batch_size=8192):
    """
    Processes expert summaries in batches, saving checkpoints after each batch.

    Parameters:
        tst_lgc_dict (dict): Dictionary containing logical group data.
        batch_size (int): Maximum token limit for each batch.

    Returns:
        dict: A dictionary of structured results.
    """
    # Prepare prompts and keys
    prompts = [
        create_prompt_expt_smry(tst_lgc_dict, key, format_instructions=expert_summary_format_instructions)
        for key in tst_lgc_dict.keys()
    ]
    keys = list(tst_lgc_dict.keys())
    
    # Batch the documents
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm.tqdm(total=len(keys), desc="Processing Expert Summaries") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    # Call get_structured_expert_summary for each key
                    structured_results[key] = get_structured_expert_summary(
                        tst_lgc_dict=tst_lgc_dict,
                        tmp_ky=key,
                        model_name="gpt-4o-mini-2024-07-18",
                        temperature=0.9
                    )
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)

    
    return structured_results

In [111]:

# Process expert summaries in batches
structured_expert_results = batch_process_expert_summaries(tst_lgc_dict)


Variables identified by the model: ['p5_1|IDE__question', 'p6|IDE__question', 'p39|FED__question']
Variables in the database: ['p5_1|IDE__question', 'p6|IDE__question', 'p39|FED__question']
Variables identified by the model: ['p1_1a_1|IDE__question', 'p2_1a_1|IDE__question']
Variables in the database: ['p1_1a_1|IDE__question', 'p2_1a_1|IDE__question']
Variables identified by the model: ['p4|IDE__question', 'p9|POB__question']
Variables in the database: ['p4|IDE__question', 'p9|POB__question']
Variables identified by the model: ['p5|IDE__question', 'p4|IDE__question']
Variables in the database: ['p4|IDE__question']
Hallucinated variables by the model: {'p5|IDE__question'}
Variables identified by the model: ['p13|IND__question', 'p14|IND__question']
Variables in the database: ['p13|IND__question', 'p14|IND__question']
Variables identified by the model: ['p2_1a_1|IDE__question', 'p7|IDE__question']
Variables in the database: ['p2_1a_1|IDE__question', 'p7|IDE__question']
Variables identifi

Processing Expert Summaries:   0%|          | 0/10 [00:00<?, ?it/s]

Variables identified by the model: ['p5_1|IDE__question', 'p6|IDE__question', 'p39|FED__question']
Variables in the database: ['p5_1|IDE__question', 'p6|IDE__question', 'p39|FED__question']


Processing Expert Summaries:  10%|█         | 1/10 [00:05<00:48,  5.44s/it]

Variables identified by the model: ['p1_1a_1|IDE__question', 'p2_1a_1|IDE__question']
Variables in the database: ['p1_1a_1|IDE__question', 'p2_1a_1|IDE__question']


...

Processing Expert Summaries:  80%|████████  | 8/10 [00:27<00:06,  3.18s/it]

Variables identified by the model: ['p17|GLO__question', 'p17|MIG__question']
Variables in the database: ['p17|MIG__question']
Hallucinated variables by the model: {'p17|GLO__question'}


Processing Expert Summaries:  90%|█████████ | 9/10 [00:31<00:03,  3.35s/it]

Variables identified by the model: ['p32|POB__question', 'p9|POB__question']
Variables in the database: ['p9|POB__question', 'p32|POB__question']


Processing Expert Summaries: 100%|██████████| 10/10 [00:34<00:00,  3.48s/it]


In [112]:
structured_expert_results

{'SIMILAR_PATTERN_1': {'EXPERT_REPLY': "The survey results reveal a significant emotional landscape regarding national identity among respondents, as indicated by the 63.42% who express 'Mucho' pride in their Mexican identity (p5_1|IDE), along with an impressive 89.25% who identify as either 'very proud' or 'proud' to be Mexican (p39|FED). These figures illustrate the strong sentiments that are crucial for discussions on national identity and policy-making, validating the concerns of experts in 'identidad y valores' who emphasize the importance of fostering pride and belonging through targeted cultural initiatives. Moreover, this emotional connection serves as a vital data point for stakeholders, including government and non-profit organizations, to tailor their programs effectively to resonate with citizens’ feelings. Furthermore, while the survey summary focuses on individual pride, insights into public perception regarding state governance and federal authority are also paramount fo

"The survey results reveal a significant emotional landscape regarding national identity among respondents, as indicated by the 63.42% who express 'Mucho' pride in their Mexican identity (p5_1|IDE), along with an impressive 89.25% who identify as either 'very proud' or 'proud' to be Mexican (p39|FED). These figures illustrate the strong sentiments that are crucial for discussions on national identity and policy-making, validating the concerns of experts in 'identidad y valores' who emphasize the importance of fostering pride and belonging through targeted cultural initiatives. Moreover, this emotional connection serves as a vital data point for stakeholders, including government and non-profit organizations, to tailor their programs effectively to resonate with citizens’ feelings. Furthermore, while the survey summary focuses on individual pride, insights into public perception regarding state governance and federal authority are also paramount for understanding the legitimacy of state

In [119]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser

# Define the Pydantic model for the output
class TransversalAnalysisResponse(BaseModel):
    TOPIC_SUMMARIES: dict[str, str]  # A dictionary with topic names as keys and summaries as values
    TOPIC_SUMMARY: str  # A one-paragraph summary for a general audience
    QUERY_ANSWER: str  # A two-sentence answer to the query

# Create the parser
transversal_parser = PydanticOutputParser(pydantic_object=TransversalAnalysisResponse)

# Generate format instructions
transversal_format_instructions = transversal_parser.get_format_instructions()

def create_prompt_trnsvl(tmp_smry_st, user_query, format_instructions=""):
    """
    Creates a prompt for analyzing expert statements and answering a query.

    Parameters:
        tmp_smry_st (str): The list of expert statements.
        user_query (str): The query to be answered.
        format_instructions (str): The format instructions generated by the PydanticOutputParser.

    Returns:
        str: The generated prompt.
    """
    prompt = f"""
    You are a very thorough research assistant and expert in survey research and public opinion.
    You are fully bilingual in English and Spanish, and can do your work in either language.

    Your task is to perform three analyses and return a single Python dictionary with the results.

    1) Read the following list of statements made by experts in several topics, which mention results, their implications, and their relevance.
    Each statement starts with the marker `*` and contains all information for a single statement: results, implications, and relevance.
    Identify three common topics across the statements and write a one-paragraph summary of each topic, citing the most relevant numbers and explanations for each.
    Format your answer as a Python dictionary with the names of the topics as keys in ALL CAPS and the summaries as values.

    Here are the statements: {tmp_smry_st}

    Save your summaries in a Python dictionary with the key `TOPIC_SUMMARIES`.

    2) Read your `TOPIC_SUMMARIES` and write a one-paragraph summary of the most relevant results and implications of the survey results, written for a general audience.
    Save your summary in a Python dictionary with the key `TOPIC_SUMMARY`.

    3) Read the QUERY and your `TOPIC_SUMMARY` and write a two-sentence answer to the QUERY that summarizes the most important points of your `TOPIC_SUMMARY`.
    Be concise and do not repeat numbers; just answer the QUERY with the relevant ideas.
    Here is the QUERY: {user_query}

    Save your answer in a Python dictionary with the key `QUERY_ANSWER`.

    IMPORTANT: Your replies for all three tasks must be in the language of the QUERY.
    IMPORTANT: Make sure to return only a correctly formatted Python dictionary, without any code block formatting, markdown, or additional text.

    {format_instructions}
    """
    return prompt


#

tmp_smry_st = ' * '.join([v['EXPERT_REPLY'] for v in structured_expert_results.values()])
tmp_smry_st

def get_transversal_analysis(tmp_smry_st, user_query, model_name='gpt-4o-mini-2024-07-18', temperature=0.9):
    # Generate the prompt
    prompt = create_prompt_trnsvl(
        tmp_smry_st=tmp_smry_st,
        user_query=user_query,
        format_instructions=transversal_format_instructions
    )
    
    # Get the response from the model
    response = get_answer(prompt=prompt, model=model_name, temperature=temperature)
    parsed = transversal_parser.parse(response)
    # Parse the response
    return parsed.model_dump()  # Returns the parsed response as a dictionary.

# Example usage
result = get_transversal_analysis(tmp_smry_st, user_query)
print(result)


{'TOPIC_SUMMARIES': {'IDENTIDAD NACIONAL Y ORGULLO': "Los resultados de la encuesta revelan una fuerte conexión emocional con la identidad nacional entre los mexicanos, destacando que el 63.42% expresa 'Mucho' orgullo por ser mexicano y el 89.25% se identifica como 'muy orgulloso' o 'orgulloso'. Esto sugiere que el orgullo nacional es un aspecto vital que puede guiar políticas culturales y de inclusión, reflejando la necesidad de fomentar la pertenencia y fortalecer la identidad entre la población.", 'DIVERSIDAD CULTURAL Y EQUIDAD': 'Una mayoría del 55.25% de los encuestados cree que una gran nación puede construirse sobre culturas distintas, lo que resalta la importancia de la diversidad cultural en la identidad nacional. Sin embargo, el 36.92% percibe diferencias entre mexicanos de primera y segunda clase, lo que indica una tensión entre la apreciación de la diversidad y los desafíos de la igualdad social, información crucial para diseñar políticas inclusivas y abordar desigualdades.

In [120]:
result

{'TOPIC_SUMMARIES': {'IDENTIDAD NACIONAL Y ORGULLO': "Los resultados de la encuesta revelan una fuerte conexión emocional con la identidad nacional entre los mexicanos, destacando que el 63.42% expresa 'Mucho' orgullo por ser mexicano y el 89.25% se identifica como 'muy orgulloso' o 'orgulloso'. Esto sugiere que el orgullo nacional es un aspecto vital que puede guiar políticas culturales y de inclusión, reflejando la necesidad de fomentar la pertenencia y fortalecer la identidad entre la población.",
  'DIVERSIDAD CULTURAL Y EQUIDAD': 'Una mayoría del 55.25% de los encuestados cree que una gran nación puede construirse sobre culturas distintas, lo que resalta la importancia de la diversidad cultural en la identidad nacional. Sin embargo, el 36.92% percibe diferencias entre mexicanos de primera y segunda clase, lo que indica una tensión entre la apreciación de la diversidad y los desafíos de la igualdad social, información crucial para diseñar políticas inclusivas y abordar desigualdade